In [1]:
import pickle
import numpy as np
from pathlib import Path
from tqdm import tqdm
from vebir.pca import loo_pca, malinowski_ind
from vebir.ebs import EBS
from vebir.veb import VEB
from vebir.metrics_utils import compute_compound_cors, compute_method_cors
import pandas as pd 
import matplotlib.pyplot as plt

In [2]:
REPO_ROOT = Path.cwd().resolve().parent
CORRECTIONS_DIR = REPO_ROOT / "data" / "corrections" / "teflon"
GT_DIR = REPO_ROOT / "data" / "spectrabase" / "teflon"
LAB_DIR = REPO_ROOT / "data" / "raw" / "teflon" / "laboratory_samples"
PREPROC_DIR = REPO_ROOT / "data" / "preprocessed" / "teflon"

with open(PREPROC_DIR / "blanks_dict.pkl", "rb") as f:
    blanks_dict = pickle.load(f)

with open(GT_DIR / "ref_dict.pkl", 'rb') as file:
        ref_dict = pickle.load(file) 

del ref_dict["CO2"]

Z = np.array(blanks_dict["2011"])
    
wn = np.sort(pd.read_csv(LAB_DIR / "zerofilling.txt", header=None).iloc[:, 0])
compounds = ['12-Tricosanone', 'Ammonium sulfate', 'Malonic Acid', 'Suberic Acid', 'D-Glucose', 'fructose', 'levoglucosan']

In [3]:
chat_ind = malinowski_ind(Z)[0]
chat_loocv = loo_pca(Z)[1]
print(f"IND: {chat_ind}, LOOCV-OSE: {chat_loocv}")

ncomp_grid = 1 + np.arange(54)
tau_grid = 0.1 * np.power(2.0, np.arange(-2, 3))
a, b = np.meshgrid(ncomp_grid, tau_grid)
grid = np.column_stack([a.ravel(), b.ravel()])

with open(CORRECTIONS_DIR / "ebs_als_dict_V.pkl", "rb") as f:
   ebs_als_dict_V = pickle.load(f)
   
with open(CORRECTIONS_DIR / "ebs_als_dict_W.pkl", "rb") as f:
   ebs_als_dict_W = pickle.load(f)
   
with open(CORRECTIONS_DIR / "ebs_pb_dict_V.pkl", "rb") as f:
   ebs_pb_dict_V = pickle.load(f)

with open(CORRECTIONS_DIR / "ebs_pb_dict_W.pkl", "rb") as f:
   ebs_pb_dict_W = pickle.load(f)

IND: 53, LOOCV-OSE: 47


In [4]:
df = {}

corrections_dict = {}
key =f"ncomp,tau = {chat_ind},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_als_dict_V[comp][key]

df["EBS-ALS-IND"] = compute_method_cors(corrections_dict,ref_dict,wn)

corrections_dict = {}
key =f"ncomp,tau = {chat_loocv},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_als_dict_W[comp][key]

df["EBS-ALS-LOOCV"] = compute_method_cors(corrections_dict,ref_dict,wn)

corrections_dict = {}
key =f"ncomp,tau = {chat_ind},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_pb_dict_V[comp][key]

df["EBS-PB-IND"] = compute_method_cors(corrections_dict,ref_dict,wn)

corrections_dict = {}
key =f"ncomp,tau = {chat_loocv},{0.10}"
for comp in compounds:
    corrections_dict[comp] = ebs_pb_dict_W[comp][key]

df["EBS-PB-LOOCV"] = compute_method_cors(corrections_dict,ref_dict,wn)

for entry in df:
    for key in df[entry]:
        df[entry][key] = f"{df[entry][key]["mean"]:.2f} pm {df[entry][key]["std_err"]:.2f}"

df = pd.DataFrame(df).T
print(df)

              12-Tricosanone Ammonium sulfate   Malonic Acid   Suberic Acid  \
EBS-ALS-IND    64.67 pm 0.14    63.06 pm 0.40  62.20 pm 0.53  58.79 pm 0.29   
EBS-ALS-LOOCV  66.77 pm 0.10    63.49 pm 0.39  63.88 pm 0.40  61.11 pm 0.25   
EBS-PB-IND     87.00 pm 0.27    77.88 pm 0.27  69.26 pm 0.74  66.72 pm 0.75   
EBS-PB-LOOCV   87.07 pm 0.27    81.25 pm 0.21  69.76 pm 0.73  67.36 pm 0.54   

                   D-Glucose       fructose   levoglucosan            all  
EBS-ALS-IND    38.65 pm 0.56  50.14 pm 0.23  48.57 pm 0.46  57.60 pm 0.57  
EBS-ALS-LOOCV  39.23 pm 0.64  53.53 pm 0.19  50.93 pm 0.59  59.38 pm 0.55  
EBS-PB-IND     68.65 pm 1.94  83.01 pm 0.89  78.80 pm 1.58  78.01 pm 0.61  
EBS-PB-LOOCV   68.69 pm 1.95  84.24 pm 0.83  81.72 pm 1.76  79.73 pm 0.63  


In [5]:
df.drop(index=["EBS-ALS-IND", "EBS-PB-IND"], inplace=True)
df.rename(index={"EBS-ALS-LOOCV":"EBS-ALS" ,"EBS-PB-LOOCV":"EBS-PB"},inplace=True)

In [6]:
df_new = {}

max_corr = 0
best_cors_dict = None

for j in range(grid.shape[0]):
    corrections_dict = {}
    params = f"ncomp,tau = {int(grid[j, 0])},{grid[j, 1]}"
    for comp in ref_dict:
        corrections_dict[comp] = ebs_als_dict_W[comp][params]
    
    cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
    new_corr = cors_dict["all"]["mean"]
    
    if new_corr > max_corr:
        max_corr = new_corr
        best_params = params
        best_cors_dict = cors_dict 

print(f"EBS-ALS best params: {best_params}")
df_new["EBS-ALS*"] = best_cors_dict

max_corr = 0
best_cors_dict = None

for j in range(grid.shape[0]):
    corrections_dict = {}
    params = f"ncomp,tau = {int(grid[j, 0])},{grid[j, 1]}"
    for comp in ref_dict:
        corrections_dict[comp] = ebs_pb_dict_W[comp][params]
    
    cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
    new_corr = cors_dict["all"]["mean"]
    
    if new_corr > max_corr:
        max_corr = new_corr
        best_params = params
        best_cors_dict = cors_dict 

print(f"EBS-ALS best params: {best_params}")
df_new["EBS-PB*"] = best_cors_dict

EBS-ALS best params: ncomp,tau = 13,0.025
EBS-ALS best params: ncomp,tau = 14,0.2


In [7]:
for entry in df_new:
    for key in df_new[entry]:
        df_new[entry][key] = f"{df_new[entry][key]['mean']:.2f} pm {df_new[entry][key]['std_err']:.2f}"

In [8]:
df = pd.concat([df,pd.DataFrame(df_new).T])
df = df.reindex(["EBS-ALS","EBS-ALS*", "EBS-PB","EBS-PB*"])

In [9]:
df

,12-Tricosanone,Ammonium sulfate,Malonic Acid,Suberic Acid,D-Glucose,fructose,levoglucosan,all
EBS-ALS,66.77 pm 0.10,63.49 pm 0.39,63.88 pm 0.40,61.11 pm 0.25,39.23 pm 0.64,53.53 pm 0.19,50.93 pm 0.59,59.38 pm 0.55
EBS-ALS*,86.26 pm 0.78,91.63 pm 0.19,70.98 pm 2.66,78.12 pm 1.56,75.22 pm 0.81,79.07 pm 1.05,80.54 pm 2.82,83.27 pm 0.81
EBS-PB,87.07 pm 0.27,81.25 pm 0.21,69.76 pm 0.73,67.36 pm 0.54,68.69 pm 1.95,84.24 pm 0.83,81.72 pm 1.76,79.73 pm 0.63
EBS-PB*,86.17 pm 0.99,97.17 pm 0.16,81.54 pm 1.47,82.51 pm 0.82,84.12 pm 2.14,82.44 pm 1.39,83.82 pm 3.10,87.33 pm 0.83


In [29]:
with open(CORRECTIONS_DIR / "veb_als_dict.pkl", "rb") as f:
   veb_als_dict = pickle.load(f)
   
with open(CORRECTIONS_DIR / "veb_als_dict_fixed.pkl", "rb") as f:
   veb_als_dict_fixed = pickle.load(f)
   
with open(CORRECTIONS_DIR / "veb_pb_dict.pkl", "rb") as f:
   veb_pb_dict = pickle.load(f)

# with open(CORRECTIONS_DIR / "veb_pb_dict_fixed.pkl", "rb") as f:
#    veb_pb_dict_fixed = pickle.load(f)

In [30]:
df = {}

corrections_dict = {}
for comp in ref_dict:
        corrections_dict[comp] = np.array([veb_als_dict[comp][i]["MAP"] for i in veb_als_dict[comp]])
        
cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
new_corr = cors_dict["all"]["mean"]

df["VEB-ALS"] = cors_dict

corrections_dict = {}
for comp in ref_dict:
        corrections_dict[comp] = np.array([veb_als_dict_fixed[comp][i]["MAP"] for i in veb_als_dict_fixed[comp]])

cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
new_corr = cors_dict["all"]["mean"]

df["VEB-ALS-Fixed"] = cors_dict

corrections_dict = {}
for comp in ref_dict:
        corrections_dict[comp] = np.array([veb_pb_dict[comp][i]["MAP"] for i in veb_pb_dict[comp]])

cors_dict =  compute_method_cors(corrections_dict, ref_dict, wn)   
new_corr = cors_dict["all"]["mean"]

df["VEB-PB"] = cors_dict

In [31]:
for entry in df:
    for key in df[entry]:
        df[entry][key] = f"{df[entry][key]["mean"]:.2f} pm {df[entry][key]["std_err"]:.2f}"

df = pd.DataFrame(df).T
print(df)

              12-Tricosanone Ammonium sulfate   Malonic Acid   Suberic Acid  \
VEB-ALS        86.02 pm 0.50    96.75 pm 0.19  77.48 pm 1.81  81.10 pm 1.03   
VEB-ALS-Fixed  86.01 pm 0.41    95.64 pm 0.30  76.31 pm 2.43  80.77 pm 1.06   
VEB-PB         86.81 pm 0.54    97.75 pm 0.14  80.16 pm 1.83  81.58 pm 1.42   

                   D-Glucose       fructose   levoglucosan            all  
VEB-ALS        83.86 pm 2.55  85.77 pm 0.99  82.72 pm 2.23  86.79 pm 0.70  
VEB-ALS-Fixed  81.36 pm 3.09  86.06 pm 0.81  79.00 pm 2.14  85.54 pm 0.72  
VEB-PB         86.93 pm 1.66  84.81 pm 1.37  86.26 pm 2.29  88.24 pm 0.70  
